In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision.transforms as transforms
import torchvision.models as models
import torch.utils.data as data
import glob
import pandas as pd
import cv2
import time
from pathlib import Path
from collections import OrderedDict
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, auc
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import timm
try:
    import umap
except ImportError:
    print("Warning: UMAP not installed. To run UMAP analysis, please `pip install umap-learn`.")
    umap = None

# ==============================================================================
# SECTION 1: DATA LOADING
# ==============================================================================

def np_load_frame(filename, resize_height, resize_width):
    image_decoded = cv2.imread(filename)
    if image_decoded is None:
        print(f"Error: Could not read image {filename}"); return None
    image_resized = cv2.resize(image_decoded, (resize_width, resize_height))
    image_resized = image_resized.astype(dtype=np.float32) / 127.5 - 1.0
    return image_resized

class DataLoader(data.Dataset):
    def __init__(self, video_folder, transform, resize_height, resize_width, time_step=4, num_pred=1, augment=False):
        self.dir, self.transform, self.augment = video_folder, transform, augment
        self._resize_height, self._resize_width = resize_height, resize_width
        self._time_step, self._num_pred = time_step, num_pred
        self.video_frames, self.index_samples = [], []
        self.aug_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor()
        ])
        self.setup()

    def setup(self):
        videos = glob.glob(os.path.join(self.dir, '*'))
        if not videos: return
        all_video_frames = []
        if os.path.isdir(videos[0]):
            for video in sorted(videos):
                all_video_frames.extend(sorted(glob.glob(os.path.join(video, '*.jpg')), key=lambda x: int(os.path.basename(x).split('.')[0].split('_')[-1])))
        else:
            all_video_frames = sorted([f for f in videos if f.lower().endswith(('.png', '.jpg', 'jpeg'))], key=lambda x: int(os.path.basename(x).split('.')[0].split('_')[-1]))
        self.video_frames = all_video_frames
        self.index_samples = list(range(len(all_video_frames) - (self._time_step + self._num_pred) + 1))

    def __getitem__(self, index):
        frame_index_start = self.index_samples[index]
        batch_frames = np.zeros((self._time_step + self._num_pred, 3, self._resize_height, self._resize_width), dtype=np.float32)
        for i in range(self._time_step + self._num_pred):
            frame = np_load_frame(self.video_frames[frame_index_start + i], self._resize_height, self._resize_width)
            if frame is not None:
                if self.augment:
                    frame_uint8 = ((frame + 1.0) * 127.5).clip(0, 255).astype(np.uint8)
                    aug_tensor = self.aug_transform(frame_uint8)
                    batch_frames[i] = (aug_tensor * 2.0 - 1.0).numpy()
                else:
                    batch_frames[i] = self.transform(frame)
        return {'standard': torch.from_numpy(batch_frames)}

    def __len__(self):
        return len(self.index_samples)

# ==============================================================================
# SECTION 2: CONFIGURATION LOGIC
# ==============================================================================

def get_train_config():
    class Config:
        exp_name = "SOTA_VAD_Advanced"
        train = 1; n_gpu = 1; image_size = 224
        num_workers = 2; lr = 1e-4; wd = 1e-4; warmup_steps = 100
        l1_weight = 0.15; ssim_weight = 0.85; perceptual_weight = 0.05
        # --- TUNABLE PARAMETERS FOR BETTER ACCURACY ---
        epochs = 20
        batch_size = 8
        unfreeze_layers = 60
        use_augmentations = True
        num_frames = 4 # Keep this consistent
    return Config()

# ==============================================================================
# SECTION 3: UTILITIES
# ==============================================================================

def setup_device(n_gpu_use):
    n_gpu = torch.cuda.device_count()
    if n_gpu_use > 0 and n_gpu == 0:
        print("Warning: There's no GPU available, training will be on CPU.")
        n_gpu_use = 0
    device = torch.device('cuda:0' if n_gpu_use > 0 else 'cpu')
    return device, list(range(n_gpu_use))

class MetricTracker:
    def __init__(self, *keys):
        self._data = pd.DataFrame(index=keys, columns=['total', 'counts', 'average'])
        self.reset()
    def reset(self):
        for col in self._data.columns: self._data[col].values[:] = 0
    def update(self, key, value, n=1):
        self._data.total[key] += value * n; self._data.counts[key] += n
        self._data.average[key] = self._data.total[key] / self._data.counts[key]
    def result(self):
        return dict(self._data.average)

# ==============================================================================
# SECTION 4: MODEL ARCHITECTURE & LOSSES
# ==============================================================================

def create_window(window_size, channel):
    _1D_window = torch.exp(-(torch.arange(window_size, dtype=torch.float) - window_size // 2)**2 / (2 * 1.5**2))
    _1D_window /= _1D_window.sum()
    _2D_window = _1D_window.unsqueeze(1) @ _1D_window.unsqueeze(0)
    return _2D_window.expand(channel, 1, window_size, window_size).contiguous()

class SSIM(nn.Module):
    def __init__(self, window_size=11, size_average=True):
        super(SSIM, self).__init__()
        self.window_size, self.size_average, self.channel = window_size, size_average, 1
        self.window = create_window(window_size, self.channel)
    def forward(self, img1, img2):
        (_, channel, _, _) = img1.size()
        if channel != self.channel or self.window.data.type() != img1.data.type() or self.window.device != img1.device:
            self.window = create_window(self.window_size, channel).to(img1.device)
            self.channel = channel
        padding = self.window_size // 2
        mu1 = F.conv2d(img1, self.window, padding=padding, groups=channel)
        mu2 = F.conv2d(img2, self.window, padding=padding, groups=channel)
        mu1_sq, mu2_sq, mu1_mu2 = mu1.pow(2), mu2.pow(2), mu1 * mu2
        sigma1_sq = F.conv2d(img1*img1, self.window, padding=padding, groups=channel) - mu1_sq
        sigma2_sq = F.conv2d(img2*img2, self.window, padding=padding, groups=channel) - mu2_sq
        sigma12 = F.conv2d(img1*img2, self.window, padding=padding, groups=channel) - mu1_mu2
        C1, C2 = 0.01**2, 0.03**2
        ssim_map = ((2*mu1_mu2 + C1)*(2*sigma12 + C2)) / ((mu1_sq + mu2_sq + C1)*(sigma1_sq + sigma2_sq + C2))
        return ssim_map.mean() if self.size_average else ssim_map.mean([1,2,3])

class PerceptualLoss(nn.Module):
    def __init__(self):
        super(PerceptualLoss, self).__init__()
        vgg = models.vgg19(pretrained=True).features
        self.slices = nn.ModuleList([vgg[i] for i in range(36)])
        for param in self.parameters(): param.requires_grad = False
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, input_img, target_img):
        input_norm, target_norm = (input_img - self.mean) / self.std, (target_img - self.mean) / self.std
        loss = 0.0
        feature_layers = {3, 8, 17, 26, 35}
        for i, layer in enumerate(self.slices):
            input_norm, target_norm = layer(input_norm), layer(target_norm)
            if i in feature_layers:
                loss += F.l1_loss(input_norm, target_norm)
        return loss

class VisionTransformerSOTA(nn.Module):
    def __init__(self, image_size=224, num_frames=4, unfreeze_layers=40):
        super(VisionTransformerSOTA, self).__init__()
        self.encoder = timm.create_model('vit_base_patch16_224', pretrained=True)
        emb_dim = self.encoder.head.in_features
        if unfreeze_layers > 0:
            for param in list(self.encoder.parameters())[:-unfreeze_layers]:
                param.requires_grad = False
        decoder_input_channels = emb_dim * num_frames
        self.decoder_net = nn.Sequential(
            nn.Conv2d(decoder_input_channels, 512, 1), nn.ConvTranspose2d(512, 256, 3, 2, 1, 1),
            nn.BatchNorm2d(256), nn.GELU(), nn.ConvTranspose2d(256, 128, 3, 2, 1, 1),
            nn.BatchNorm2d(128), nn.GELU(), nn.ConvTranspose2d(128, 64, 3, 2, 1, 1),
            nn.BatchNorm2d(64), nn.GELU(), nn.ConvTranspose2d(64, 3, 3, 2, 1, 1), nn.Tanh())
    def forward(self, x):
        batch_size, num_frames, C, H, W = x.shape
        feature_map_size = H // 16
        frame_features_list = [self.encoder.forward_features(x[:, i])[:, 1:].permute(0, 2, 1).reshape(batch_size, -1, feature_map_size, feature_map_size) for i in range(num_frames)]
        return self.decoder_net(torch.cat(frame_features_list, dim=1))
    def forward_features(self, x):
        return self.encoder.forward_features(x)

# ==============================================================================
# SECTION 5: VISUALIZATION & ANALYSIS
# ==============================================================================
def save_anomaly_visualization(gt_frame, pred_frame, epoch, batch_idx, save_dir):
    gt_img = (gt_frame.squeeze(0).permute(1, 2, 0).cpu().numpy() + 1.0) * 127.5
    pred_img = (pred_frame.squeeze(0).permute(1, 2, 0).cpu().numpy() + 1.0) * 127.5
    residual_color = cv2.applyColorMap(cv2.cvtColor(cv2.absdiff(gt_img, pred_img).astype(np.uint8), cv2.COLOR_BGR2GRAY), cv2.COLORMAP_JET)
    combined_img = np.concatenate((gt_img.astype(np.uint8), pred_img.astype(np.uint8), residual_color), axis=1)
    cv2.imwrite(os.path.join(save_dir, f"epoch_{epoch}_frame_{batch_idx}.jpg"), combined_img)

def save_roc_curves(history, best_epoch_data, output_dir):
    if not history: return
    plt.figure(figsize=(12, 9))
    colors = plt.cm.viridis(np.linspace(0, 1, len(history)))
    for epoch_data in history:
        plt.plot(epoch_data['fpr'], epoch_data['tpr'], lw=1.5, color=colors[epoch_data['epoch']-1], label=f"Epoch {epoch_data['epoch']} (AUC = {epoch_data['auc']:.4f})")
    plt.plot([0, 1], [0, 1], 'k--', label='Chance'); plt.title('Model Progression: ROC Curves'); plt.xlabel('FPR'); plt.ylabel('TPR')
    plt.legend(); plt.grid(True); plt.savefig(os.path.join(output_dir, 'all_epochs_roc_curve.jpg')); plt.close()
    if best_epoch_data:
        plt.figure(figsize=(6, 6))
        plt.plot(best_epoch_data['fpr'], best_epoch_data['tpr'], lw=2, color='deeppink', label=f"Ours (AUC = {best_epoch_data['auc']:.4f})")
        plt.plot([0, 1], [0, 1], 'k--'); plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.0])
        plt.title('ROC on UIT-ADrone dataset'); plt.xlabel('FPR'); plt.ylabel('TPR')
        plt.legend(); plt.grid(False); plt.savefig(os.path.join(output_dir, 'publication_style_roc_curve.jpg'), dpi=300); plt.close()

def analyze_frequency_domain(model, data_loader, device, output_dir, config):
    print("\n--- Starting Frequency Domain (FFT) Analysis ---")
    model.eval()
    label_path = '/kaggle/input/npy-dataset11/50m_90d_morning_congkhuA_22_3.npy'
    if not os.path.exists(label_path): return
    ground_truth_labels = np.load(label_path, allow_pickle=True)
    normal_spectrums, anomalous_spectrums = [], []
    with torch.no_grad():
        for i, batch_data in enumerate(tqdm(data_loader, desc="FFT Analysis")):
            inputs, target = batch_data['standard'][:, :config.num_frames].to(device), batch_data['standard'][:, config.num_frames].to(device)
            try: label = ground_truth_labels[int(os.path.basename(data_loader.dataset.video_frames[i + config.num_frames]).split('.')[0].split('_')[-1])]
            except: label = 0
            pred = model(inputs)
            residual_gray = torch.mean(torch.abs(target - pred), dim=1).squeeze(0).cpu().numpy()
            f_transform_shifted = np.fft.fftshift(np.fft.fft2(residual_gray))
            magnitude_spectrum = 20 * np.log(np.abs(f_transform_shifted) + 1)
            (anomalous_spectrums if label == 1 else normal_spectrums).append(magnitude_spectrum)
    if not normal_spectrums or not anomalous_spectrums: return
    avg_normal, avg_anomalous = np.mean(normal_spectrums, axis=0), np.mean(anomalous_spectrums, axis=0)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6)); fig.suptitle('Frequency Domain Analysis of Residual Maps', fontsize=16)
    axes[0].imshow(avg_normal, cmap='viridis'); axes[0].set_title('Average Spectrum of Normal Residuals')
    axes[1].imshow(avg_anomalous, cmap='viridis'); axes[1].set_title('Average Spectrum of Anomalous Residuals')
    diff = avg_anomalous - avg_normal; vlim = np.abs(diff).max()
    im = axes[2].imshow(diff, cmap='coolwarm', vmin=-vlim, vmax=vlim); axes[2].set_title('Difference (Anomalous - Normal)')
    fig.colorbar(im, ax=axes[2]); plt.tight_layout(rect=[0,0,1,0.95])
    plt.savefig(os.path.join(output_dir, 'fft_analysis_2D.jpg')); plt.close()
    print("Saved 2D FFT analysis plot.")
    plot_radial_profile(normal_spectrums, anomalous_spectrums, output_dir)
    print("--- FFT Analysis Finished ---")

def plot_radial_profile(normal_spectrums, anomalous_spectrums, output_dir):
    def calculate_profiles(spectrums):
        h, w = spectrums[0].shape; center = (h//2, w//2)
        y, x = np.indices((h, w)); distances = np.sqrt((x-center[1])**2 + (y-center[0])**2).astype(int)
        all_profiles = []
        for s in spectrums:
            counts = np.bincount(distances.ravel())
            sums = np.bincount(distances.ravel(), weights=s.ravel())
            all_profiles.append(sums / np.where(counts > 0, counts, 1))
        max_len = max(len(p) for p in all_profiles)
        padded = [np.pad(p, (0, max_len - len(p)), mode='edge') for p in all_profiles]
        return np.mean(padded, axis=0), np.std(padded, axis=0)
    mean_n, std_n = calculate_profiles(normal_spectrums)
    mean_a, std_a = calculate_profiles(anomalous_spectrums)
    plt.figure(figsize=(12, 7)); x_axis = np.arange(len(mean_n))
    plt.plot(x_axis, mean_n, 'b-', label='Normal Residuals (Mean)'); plt.fill_between(x_axis, mean_n-std_n, mean_n+std_n, color='b', alpha=0.2)
    plt.plot(x_axis, mean_a, 'r-', label='Anomalous Residuals (Mean)'); plt.fill_between(x_axis, mean_a-std_a, mean_a+std_a, color='r', alpha=0.2)
    plt.yscale('log'); plt.xscale('log'); plt.title('1D Radially Averaged FFT Spectrum with Error Bands'); plt.xlabel('Frequency (pixels from center, log)'); plt.ylabel('Avg Log Magnitude (log)')
    plt.legend(); plt.grid(True, which="both", ls="--"); plt.savefig(os.path.join(output_dir, 'fft_radial_profile.jpg')); plt.close()
    print("Saved 1D radial profile plot.")

def analyze_embedding_separation(model, data_loader, device, output_dir, config, reducer_type='tsne', num_samples_per_class=50):
    if reducer_type == 'umap' and umap is None: return
    print(f"\n--- Starting Embedding Separation ({reducer_type.upper()}) Analysis ---")
    model.eval()
    layer_indices = {'Early': 1, 'Middle': 5, 'Late': 11}; activations = {}
    def get_activation(name):
        def hook(model, input, output): activations[name] = output[:, 0, :].detach().cpu()
        return hook
    hooks = {}; model_to_hook = model.module if hasattr(model, 'module') else model
    for name, idx in layer_indices.items(): hooks[name] = model_to_hook.encoder.blocks[idx].register_forward_hook(get_activation(name))
    features = {name: [] for name in layer_indices}; labels_list = []
    label_path = '/kaggle/input/npy-dataset11/50m_90d_morning_congkhuA_22_3.npy'
    if not os.path.exists(label_path): return
    ground_truth_labels = np.load(label_path, allow_pickle=True)
    normal_collected, anomalous_collected = 0, 0
    for i, batch_data in enumerate(tqdm(data_loader, desc=f"Collecting for {reducer_type.upper()}")):
        if normal_collected >= num_samples_per_class and anomalous_collected >= num_samples_per_class: break
        try: label = ground_truth_labels[int(os.path.basename(data_loader.dataset.video_frames[i + config.num_frames]).split('.')[0].split('_')[-1])]
        except: label = 0
        if (label == 0 and normal_collected >= num_samples_per_class) or (label == 1 and anomalous_collected >= num_samples_per_class): continue
        target_frame = batch_data['standard'][:, config.num_frames].to(device)
        _ = model_to_hook.forward_features(target_frame)
        for name in layer_indices: features[name].append(activations[name])
        labels_list.append(label); normal_collected += (1-label); anomalous_collected += label
    for hook in hooks.values(): hook.remove()
    if normal_collected < 10 or anomalous_collected < 10: return
    print(f"Collected {normal_collected} normal and {anomalous_collected} anomalous samples.")
    print("--- Quantitative Separability ---")
    y = np.array(labels_list)
    for name in layer_indices:
        X = torch.cat(features[name]).numpy()
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
        acc = LogisticRegression(random_state=42, max_iter=1000).fit(X_train, y_train).score(X_test, y_test)
        print(f"[{name} Layer] Accuracy: {acc:.2%}")
    fig, axes = plt.subplots(1, 3, figsize=(20, 6)); fig.suptitle(f'{reducer_type.upper()} Visualization by Layer', fontsize=16)
    labels_arr = np.array(labels_list)
    for ax, name in zip(axes, layer_indices):
        X = torch.cat(features[name]).numpy()
        if reducer_type == 'tsne':
            reducer = TSNE(n_components=2, perplexity=min(30, len(X)-1), learning_rate='auto', init='pca', random_state=42)
        else:
            reducer = umap.UMAP(n_neighbors=min(15, len(X)-1), min_dist=0.1, n_components=2, random_state=42)
        embeddings = reducer.fit_transform(X)
        for label_val, color, marker in zip([0, 1], ['blue', 'red'], ['o', 'x']):
            idx = labels_arr == label_val
            ax.scatter(embeddings[idx, 0], embeddings[idx, 1], c=color, label=f'{"Normal" if label_val==0 else "Anomalous"}', marker=marker, alpha=0.7)
        ax.set_title(f'{name} Layer'); ax.legend()
    plt.savefig(os.path.join(output_dir, f'embedding_separation_{reducer_type}.jpg')); plt.close()
    print(f"Saved {reducer_type.upper()} plot.")
    print(f"--- {reducer_type.upper()} Analysis Finished ---")

def analyze_attention_maps(model, data_loader, device, output_dir, config, num_samples=5):
    print("\n--- Starting Anomaly Localization (Attention Map) Analysis ---")
    model.eval()
    label_path = '/kaggle/input/npy-dataset11/50m_90d_morning_congkhuA_22_3.npy'
    if not os.path.exists(label_path): return
    ground_truth_labels = np.load(label_path, allow_pickle=True)
    attention_weights = {}
    def get_attention(name):
        def hook(model, input, output): attention_weights[name] = model.attn.get_attention_map()
        return hook
    model_to_hook = model.module if hasattr(model, 'module') else model
    hook = model_to_hook.encoder.blocks[-1].attn.register_forward_hook(get_attention('last_block_attn'))
    anomalous_found = 0
    with torch.no_grad():
        for i, batch_data in enumerate(tqdm(data_loader, desc="Finding Anomalies for Attention Viz")):
            if anomalous_found >= num_samples: break
            try: label = ground_truth_labels[int(os.path.basename(data_loader.dataset.video_frames[i + config.num_frames]).split('.')[0].split('_')[-1])]
            except: label = 0
            if label == 1:
                target_frame = batch_data['standard'][:, config.num_frames].to(device)
                _ = model_to_hook.forward_features(target_frame)
                attn_map = attention_weights['last_block_attn'].squeeze(0)
                cls_attn = attn_map[:, 0, 1:].mean(dim=0)
                patch_size = model_to_hook.encoder.patch_embed.patch_size[0]
                grid_size = config.image_size // patch_size
                mask = cls_attn.reshape(grid_size, grid_size).cpu().numpy()
                mask = cv2.resize(mask / mask.max(), (config.image_size, config.image_size))
                img = (target_frame.squeeze(0).permute(1, 2, 0).cpu().numpy() + 1.0) * 127.5
                heatmap = cv2.applyColorMap(np.uint8(255 * mask), cv2.COLORMAP_JET)
                overlaid_img = cv2.addWeighted(np.uint8(img), 0.6, heatmap, 0.4, 0)
                combined_img = np.concatenate((np.uint8(img), heatmap, overlaid_img), axis=1)
                cv2.imwrite(os.path.join(output_dir, f"attention_map_anomaly_{anomalous_found}.jpg"), combined_img)
                anomalous_found += 1
    hook.remove()
    print(f"Saved {anomalous_found} attention map visualizations.")

def evaluate_frequency_score(model, data_loader, device, output_dir, config, low_freq=10, high_freq=100):
    print("\n--- Evaluating Frequency-Based Anomaly Score ---")
    model.eval()
    label_path = '/kaggle/input/npy-dataset11/50m_90d_morning_congkhuA_22_3.npy'
    if not os.path.exists(label_path): return
    ground_truth_labels = np.load(label_path, allow_pickle=True)
    all_freq_scores, all_labels = [], []
    with torch.no_grad():
        for i, batch_data in enumerate(tqdm(data_loader, desc="Evaluating Frequency Score")):
            inputs, target = batch_data['standard'][:, :config.num_frames].to(device), batch_data['standard'][:, config.num_frames].to(device)
            try: label = ground_truth_labels[int(os.path.basename(data_loader.dataset.video_frames[i + config.num_frames]).split('.')[0].split('_')[-1])]
            except: label = 0
            pred = model(inputs)
            residual_gray = torch.mean(torch.abs(target - pred), dim=1).squeeze(0).cpu().numpy()
            magnitude_spectrum = np.abs(np.fft.fftshift(np.fft.fft2(residual_gray)))
            h, w = magnitude_spectrum.shape; center = (h//2, w//2)
            y, x = np.indices((h, w)); distances = np.sqrt((x-center[1])**2 + (y-center[0])**2)
            mask = (distances >= low_freq) & (distances <= high_freq)
            freq_score = magnitude_spectrum[mask].mean()
            all_freq_scores.append(freq_score); all_labels.append(label)
    if len(np.unique(all_labels)) < 2: return
    auc_score = roc_auc_score(all_labels, all_freq_scores)
    print(f"AUC using Frequency-Band Score ({low_freq}-{high_freq} pixels): {auc_score:.4f}")
    precision, recall, _ = precision_recall_curve(all_labels, all_freq_scores)
    pr_auc = auc(recall, precision)
    plt.figure(figsize=(8,6)); plt.plot(recall, precision, label=f'PR Curve (AUC = {pr_auc:.4f})')
    plt.title('PR Curve for Frequency Score'); plt.xlabel('Recall'); plt.ylabel('Precision'); plt.legend(); plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'frequency_score_pr_curve.jpg')); plt.close()
    print("Saved PR curve for frequency score.")

# ==============================================================================
# SECTION 6: TRAINING & EVALUATION LOGIC
# ==============================================================================
def train_epoch(epoch, model, data_loader, optimizer, lr_scheduler, metrics, device, config):
    model.train(); metrics.reset()
    loss_l1, loss_ssim, loss_p = nn.L1Loss().to(device), SSIM().to(device), PerceptualLoss().to(device)
    for batch_data in tqdm(data_loader, desc=f"Training Epoch {epoch}"):
        inputs, target = batch_data['standard'][:,:config.num_frames].to(device), batch_data['standard'][:,config.num_frames].to(device)
        optimizer.zero_grad()
        pred = model(inputs)
        pred_norm, target_norm = (pred+1)/2, (target+1)/2
        loss = config.l1_weight*loss_l1(pred, target) + config.ssim_weight*(1-loss_ssim(pred_norm, target_norm)) + config.perceptual_weight*loss_p(pred_norm, target_norm)
        loss.backward(); optimizer.step()
        if lr_scheduler: lr_scheduler.step()
        metrics.update('loss', loss.item())
    print(f"Train Epoch: {epoch:03d} Avg Loss: {metrics.result()['loss']:.4f}")
    return metrics.result()

def valid_epoch(epoch, model, data_loader, metrics, device, frame_save_dir, config):
    model.eval(); metrics.reset()
    all_anomaly_scores, all_labels = [], []
    loss_l1, loss_ssim = nn.L1Loss(reduction='none').to(device), SSIM(size_average=False).to(device)
    label_path = '/kaggle/input/npy-dataset11/50m_90d_morning_congkhuA_22_3.npy'
    if not os.path.exists(label_path): return {'loss':-1, 'auc':0.0, 'eer':1.0, 'fpr':np.array([0,1]), 'tpr':np.array([0,1])}
    ground_truth_labels = np.load(label_path, allow_pickle=True)
    with torch.no_grad():
        for i, batch_data in enumerate(tqdm(data_loader, desc=f"Validating Epoch {epoch}")):
            inputs, target = batch_data['standard'][:,:config.num_frames].to(device), batch_data['standard'][:,config.num_frames].to(device)
            pred = model(inputs)
            pred_norm, target_norm = (pred+1)/2, (target+1)/2
            anomaly_scores = (1 - config.ssim_weight) * torch.mean(loss_l1(pred, target).view(target.size(0),-1), dim=1) + config.ssim_weight * (1 - loss_ssim(pred_norm, target_norm))
            all_anomaly_scores.extend(anomaly_scores.cpu().numpy())
            try:
                frame_num = int(os.path.basename(data_loader.dataset.video_frames[i + config.num_frames]).split('.')[0].split('_')[-1])
                all_labels.append(ground_truth_labels[frame_num] if frame_num < len(ground_truth_labels) else 0)
            except: all_labels.append(0)
            if i < 5: save_anomaly_visualization(target[0:1], pred[0:1], epoch, i, frame_save_dir)
    final_labels = all_labels[:len(all_anomaly_scores)]
    if len(np.unique(final_labels)) < 2: return {'loss':-1, 'auc':0.0, 'eer':1.0, 'fpr':np.array([0,1]), 'tpr':np.array([0,1])}
    auc_score = roc_auc_score(final_labels, all_anomaly_scores)
    fpr, tpr, _ = roc_curve(final_labels, all_anomaly_scores)
    fnr = 1-tpr; eer_idx = np.nanargmin(np.abs(fnr-fpr)); eer = fpr[eer_idx]
    metrics.update('loss', np.mean(all_anomaly_scores)); metrics.update('auc', auc_score); metrics.update('eer', eer)
    print(f"Validation Epoch: {epoch:03d}, AUC: {auc_score:.4f}, EER: {eer:.4f}")
    return {'loss': metrics.result()['loss'], 'auc': auc_score, 'eer': eer, 'fpr': fpr, 'tpr': tpr}

def save_model(epoch, model, optimizer, lr_scheduler, device_ids, best=False, save_dir='outputs'):
    state = {'epoch':epoch, 'state_dict':model.state_dict(), 'optimizer':optimizer.state_dict(), 'lr_scheduler':lr_scheduler.state_dict() if lr_scheduler else None}
    if len(device_ids) > 1: state['state_dict'] = model.module.state_dict()
    checkpoints_dir = os.path.join(save_dir, 'checkpoints')
    os.makedirs(checkpoints_dir, exist_ok=True)
    torch.save(state, os.path.join(checkpoints_dir, 'current.pth'))
    if best: torch.save(state, os.path.join(checkpoints_dir, 'best.pth'))

# ==============================================================================
# SECTION 7: MAIN EXECUTION BLOCK
# ==============================================================================
def main():
    config = get_train_config()
    device, device_ids = setup_device(config.n_gpu)
    train_metrics, valid_metrics = MetricTracker('loss'), MetricTracker('loss', 'auc', 'eer')
    output_dir = "/kaggle/working/"; os.makedirs(os.path.join(output_dir, "output_frames"), exist_ok=True)
    log_file_path = os.path.join(output_dir, 'training_log.csv')
    with open(log_file_path, 'w') as log_file: log_file.write('epoch,train_loss,val_loss,val_auc,val_eer\n')
    
    model = VisionTransformerSOTA(config.image_size, config.num_frames, config.unfreeze_layers).to(device)
    if len(device_ids) > 1: model = nn.DataParallel(model, device_ids=device_ids)
    
    if bool(config.train):
        train_folder, test_folder = "/kaggle/input/dataset1/archive/50m_90d_morning_congkhuA_22_3_train", "/kaggle/input/dataset1/archive/50m_90d_morning_congkhuA_22_3_test"
        data_transform = transforms.Compose([transforms.ToTensor()])
        train_dataset = DataLoader(train_folder, data_transform, config.image_size, config.image_size, config.num_frames, augment=config.use_augmentations)
        train_batch = data.DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=config.num_workers, drop_last=True)
        test_dataset = DataLoader(test_folder, data_transform, config.image_size, config.image_size, config.num_frames)
        test_batch = data.DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=config.num_workers)
        
        optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.wd)
        total_steps = len(train_batch) * config.epochs
        warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=config.warmup_steps)
        main_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps-config.warmup_steps, eta_min=1e-6)
        lr_scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, main_scheduler], milestones=[config.warmup_steps])
        
        best_auc = 0.0; roc_history = []; best_epoch_data = None
        for epoch in range(1, config.epochs + 1):
            train_result = train_epoch(epoch, model, train_batch, optimizer, lr_scheduler, train_metrics, device, config)
            valid_result = valid_epoch(epoch, model, test_batch, valid_metrics, device, os.path.join(output_dir, "output_frames"), config)
            roc_history.append({'epoch': epoch, **valid_result})
            with open(log_file_path, 'a') as log_file: log_file.write(f"{epoch},{train_result['loss']:.6f},{valid_result['loss']:.6f},{valid_result['auc']:.6f},{valid_result['eer']:.6f}\n")
            if valid_result['auc'] > best_auc:
                best_auc = valid_result['auc']; best_epoch_data = {'epoch': epoch, **valid_result}
                print(f"*** New best AUC: {best_auc:.4f} at epoch {epoch}. Saving best model. ***")
                save_model(epoch, model, optimizer, lr_scheduler, device_ids, best=True, save_dir=output_dir)
            save_model(epoch, model, optimizer, lr_scheduler, device_ids, best=False, save_dir=output_dir)
        print("\n--- Training finished. ---")
        
        save_roc_curves(roc_history, best_epoch_data, output_dir)
        print("Saved ROC curve plots.")

        if best_epoch_data:
            best_model_path = os.path.join(output_dir, 'checkpoints', 'best.pth')
            if os.path.exists(best_model_path):
                print(f"\nLoading best model from epoch {best_epoch_data['epoch']} for final analysis.")
                # We need to pass the config value for unfreeze_layers here
                analysis_model = VisionTransformerSOTA(config.image_size, config.num_frames, config.unfreeze_layers).to(device)
                checkpoint = torch.load(best_model_path, map_location=device)
                state_dict = checkpoint['state_dict']
                if list(state_dict.keys())[0].startswith('module.'):
                    new_state_dict = OrderedDict([(k[7:], v) for k, v in state_dict.items()])
                    analysis_model.load_state_dict(new_state_dict)
                else:
                    analysis_model.load_state_dict(state_dict)

                analyze_frequency_domain(analysis_model, test_batch, device, output_dir, config)
                analyze_embedding_separation(analysis_model, test_batch, device, output_dir, config, reducer_type='tsne')
                analyze_embedding_separation(analysis_model, test_batch, device, output_dir, config, reducer_type='umap')
                analyze_attention_maps(analysis_model, test_batch, device, output_dir, config)
                evaluate_frequency_score(analysis_model, test_batch, device, output_dir, config)

if __name__ == '__main__':
    main()

2025-09-22 12:53:32.292725: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758545612.464113      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758545612.519267      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:02<00:00, 201MB/s] 
Training Epoch 1:   0%|          | 0/110 [00:00<?, ?it/s]/tmp/ipykernel_36/2402687581.py:120: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chai

Train Epoch: 001 Avg Loss: 0.8610


Validating Epoch 1: 100%|██████████| 2349/2349 [02:30<00:00, 15.65it/s]
/tmp/ipykernel_36/2402687581.py:120: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self._data.total[key] += value * n; self._data.counts[key] += n
/tmp/ipykernel_36/2402

...

Validating Epoch 7: 100%|██████████| 2349/2349 [02:18<00:00, 16.97it/s]
/tmp/ipykernel_36/2402687581.py:120: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  self._data.total[key] += value * n; self._data.counts[key] += n
/tmp/ipykernel_36/2402

Validation Epoch: 007, AUC: 0.6446, EER: 0.4054


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Training Epoch 8: 100%|██████████| 110/110 [01:04<00:00,  1.72it/s]


Train Epoch: 008 Avg Loss: 0.6367


Validating Epoch 8:  52%|█████▏    | 1215/2349 [01:12<01:04, 17.55it/s]